# Phase 5 - NLLB-200 + MorphBPE Kapampangan->Filipino fine-tune

**You run this in Colab. Claude Code wrote it and cannot run it.**

Compares three source-tokenizer conditions for adapting
`facebook/nllb-200-distilled-600M` to Kapampangan->Filipino, in the
low-resource regime this project actually has (598 training pairs, ALL
SILVER):

| condition | encoder source tokenizer | what trains |
|---|---|---|
| `nllb_native` | NLLB's own SentencePiece (Kapampangan read as `tgl_Latn`) | encoder input embedding only (untied from `shared`) |
| `morphbpe` | paper-aligned hard-constrained MorphBPE, vocab 6,080 | fresh `nn.Embedding(6080, 1024)` only |
| `penalty8` | weighted MorphBPE (crossing penalty 8), vocab 6,080 | fresh `nn.Embedding(6080, 1024)` only |

Everything else in the model stays **frozen**, and `tie_weights()` is never
called after the swap (Phase 4 finding, `nllb/phase4-architecture-verification.md`).
Each condition runs on **3 seeds** (0, 1, 2). A `nllb_zeroshot` reference
(no training) is measured once.

### How to run
1. `Runtime -> Change runtime type -> T4 GPU`.
2. Upload the bundle: the cell below expects `train.jsonl`, `dev.jsonl`,
   `test.jsonl`, `meta.json` from
   `experiments/nllb_finetune_v1/data/bundle/` in the Colab working dir
   (drag them into the file panel, or mount Drive).
3. `Runtime -> Run all`. ~2-2.5 h on a T4 (10 runs). Results are written
   incrementally to `phase5-results.json`; re-running skips finished runs,
   so a disconnect is recoverable.
4. Download `phase5-results.json` and send it back.

Data note: the training pairs are PLD-derived (redistribution rights
unresolved) + native-authored stories + `gold_v1`. This notebook only
uploads them into your Colab runtime; nothing is published.

In [ ]:
%pip install -q -U "transformers>=4.44,<5" sentencepiece sacremoses sacrebleu
import json, math, os, random, time, platform
import numpy as np
import torch, torch.nn as nn
import transformers, sacrebleu
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
print("transformers", transformers.__version__, "| torch", torch.__version__, "| sacrebleu", sacrebleu.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print("GPU:", p.name, round(p.total_memory/1024**3, 1), "GiB")
else:
    print("WARNING: no GPU - this will be very slow.")

In [ ]:
# --- load the bundle ---
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

for fn in ["train.jsonl", "dev.jsonl", "test.jsonl", "meta.json"]:
    assert os.path.exists(fn), f"missing {fn} - upload the bundle files first"

META = json.load(open("meta.json", encoding="utf-8"))
TRAIN = read_jsonl("train.jsonl"); DEV = read_jsonl("dev.jsonl"); TEST = read_jsonl("test.jsonl")
print(f"train {len(TRAIN)} | dev {len(DEV)} | test {len(TEST)}")
print("meta:", json.dumps({k: META[k] for k in ("conditions","seeds","source_vocab_size_morphbpe","nllb")}, indent=1))

TGT_LANG = META["nllb"]["target_lang"]            # tgl_Latn
NATIVE_SRC_LANG = META["nllb"]["native_baseline_source_lang"]
SRC_VOCAB = META["source_vocab_size_morphbpe"]    # 6080
SRC_PAD = META["source_pad_id_morphbpe"]          # 0
MODEL_NAME = META["nllb"]["model"]

RESULTS_PATH = "phase5-results.json"
results = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}
def save_results():
    json.dump(results, open(RESULTS_PATH, "w"), indent=2, ensure_ascii=False)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.tgt_lang = TGT_LANG
TGT_BOS = tokenizer.convert_tokens_to_ids(TGT_LANG)
PAD = tokenizer.pad_token_id

def fresh_model():
    m = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
    m.config.use_cache = False
    return m

# --- source-side id builders per condition ---
def src_ids_for(condition, rec):
    if condition == "morphbpe":
        return rec["morphbpe_ids"]
    if condition == "penalty8":
        return rec["penalty8_ids"]
    # nllb_native / nllb_zeroshot: tokenise pam_text with NLLB (src=tgl_Latn)
    tokenizer.src_lang = NATIVE_SRC_LANG
    return tokenizer(rec["pam_text"], add_special_tokens=True)["input_ids"]

def label_ids(rec):
    ids = tokenizer(text_target=rec["fil_text"], add_special_tokens=True)["input_ids"]
    return ids

def collate(batch, condition, src_pad):
    src = [src_ids_for(condition, r) for r in batch]
    lab = [label_ids(r) for r in batch]
    sm = max(len(x) for x in src); lm = max(len(x) for x in lab)
    input_ids = torch.full((len(batch), sm), src_pad, dtype=torch.long)
    attn = torch.zeros((len(batch), sm), dtype=torch.long)
    labels = torch.full((len(batch), lm), -100, dtype=torch.long)
    for i,(s,l) in enumerate(zip(src, lab)):
        input_ids[i,:len(s)] = torch.tensor(s); attn[i,:len(s)] = 1
        labels[i,:len(l)] = torch.tensor(l)
    return input_ids.to(DEVICE), attn.to(DEVICE), labels.to(DEVICE)

In [ ]:
# --- the encoder-embedding swap (Phase 4 recipe) ---
def prepare_model(condition, seed):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    m = fresh_model()
    core = m.model
    d = m.config.d_model
    for p in m.parameters():
        p.requires_grad_(False)

    if condition in ("morphbpe", "penalty8"):
        emb = nn.Embedding(SRC_VOCAB, d, padding_idx=SRC_PAD)
        nn.init.normal_(emb.weight, mean=0.0, std=d ** -0.5)
        with torch.no_grad():
            emb.weight[SRC_PAD].zero_()
    else:  # nllb_native: untie a trainable copy of the encoder embedding
        emb = nn.Embedding(core.shared.num_embeddings, d, padding_idx=core.shared.padding_idx)
        with torch.no_grad():
            emb.weight.copy_(core.shared.weight)

    emb.weight.requires_grad_(True)
    core.encoder.embed_tokens = emb          # encoder ONLY; never call tie_weights() now
    m.to(DEVICE)
    trainable = [p for p in m.parameters() if p.requires_grad]
    assert len(trainable) == 1 and trainable[0] is emb.weight
    return m, emb

In [ ]:
BATCH = 8
EPOCHS = 30
PATIENCE = 6
LR = 1e-3
GEN_KW = dict(num_beams=5, max_new_tokens=96, forced_bos_token_id=TGT_BOS)

def batched(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

@torch.no_grad()
def evaluate(m, data, condition):
    m.eval()
    hyps, refs = [], [r["fil_text"] for r in data]
    for chunk in batched(data, BATCH):
        src = [src_ids_for(condition, r) for r in chunk]
        sm = max(len(x) for x in src)
        pad = SRC_PAD if condition in ("morphbpe","penalty8") else PAD
        ii = torch.full((len(chunk), sm), pad, dtype=torch.long)
        am = torch.zeros((len(chunk), sm), dtype=torch.long)
        for i,s in enumerate(src):
            ii[i,:len(s)] = torch.tensor(s); am[i,:len(s)] = 1
        out = m.generate(input_ids=ii.to(DEVICE), attention_mask=am.to(DEVICE), **GEN_KW)
        hyps += tokenizer.batch_decode(out, skip_special_tokens=True)
    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
    chrf = sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score   # chrF++
    return {"bleu": round(bleu,2), "chrf": round(chrf,2)}, hyps

def train_one(condition, seed):
    key = f"{condition}/seed{seed}"
    if key in results:
        print("skip (done):", key); return
    t0 = time.time()
    src_pad = SRC_PAD if condition in ("morphbpe","penalty8") else PAD
    m, emb = prepare_model(condition, seed)
    opt = torch.optim.AdamW([emb.weight], lr=LR)
    order = list(range(len(TRAIN)))
    best = {"chrf": -1}; best_state = None; bad = 0
    for ep in range(1, EPOCHS+1):
        m.train(); random.Random(1000+seed*97+ep).shuffle(order)
        tot = 0.0; nb = 0
        for idx in batched(order, BATCH):
            batch = [TRAIN[i] for i in idx]
            ii, am, lab = collate(batch, condition, src_pad)
            loss = m(input_ids=ii, attention_mask=am, labels=lab).loss
            loss.backward(); opt.step(); opt.zero_grad()
            tot += float(loss); nb += 1
        dev_score, _ = evaluate(m, DEV, condition)
        print(f"  {key} ep{ep:02d} loss {tot/nb:.3f}  dev chrF++ {dev_score['chrf']}  BLEU {dev_score['bleu']}")
        if dev_score["chrf"] > best["chrf"]:
            best = {**dev_score, "epoch": ep}; best_state = emb.weight.detach().clone(); bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print("  early stop"); break
    with torch.no_grad():
        emb.weight.copy_(best_state)
    test_score, test_hyps = evaluate(m, TEST, condition)
    results[key] = {
        "condition": condition, "seed": seed,
        "best_epoch": best["epoch"], "dev": {"chrf": best["chrf"], "bleu": best["bleu"]},
        "test": test_score, "minutes": round((time.time()-t0)/60, 1),
    }
    save_results()
    print(f"  -> {key}  TEST chrF++ {test_score['chrf']}  BLEU {test_score['bleu']}  ({results[key]['minutes']} min)")
    del m, emb; torch.cuda.empty_cache()

In [ ]:
# --- zero-shot reference (no training), once ---
if "nllb_zeroshot" not in results:
    m = fresh_model().to(DEVICE)
    sc, _ = evaluate(m, TEST, "nllb_zeroshot")
    dv, _ = evaluate(m, DEV, "nllb_zeroshot")
    results["nllb_zeroshot"] = {"condition": "nllb_zeroshot", "dev": dv, "test": sc}
    save_results(); del m; torch.cuda.empty_cache()
    print("nllb_zeroshot TEST", sc)

# --- the 9 training runs ---
for condition in ["nllb_native", "morphbpe", "penalty8"]:
    for seed in [0, 1, 2]:
        train_one(condition, seed)

In [ ]:
# --- summary: mean +/- std across seeds, write + download ---
from statistics import mean, pstdev
summary = {"per_run": results, "aggregate": {}}
for condition in ["nllb_native", "morphbpe", "penalty8"]:
    runs = [results[f"{condition}/seed{s}"] for s in [0,1,2] if f"{condition}/seed{s}" in results]
    if not runs: continue
    tc = [r["test"]["chrf"] for r in runs]; tb = [r["test"]["bleu"] for r in runs]
    summary["aggregate"][condition] = {
        "n_seeds": len(runs),
        "test_chrf_mean": round(mean(tc),2), "test_chrf_std": round(pstdev(tc),2),
        "test_bleu_mean": round(mean(tb),2), "test_bleu_std": round(pstdev(tb),2),
    }
if "nllb_zeroshot" in results:
    summary["aggregate"]["nllb_zeroshot"] = results["nllb_zeroshot"]["test"]
summary["meta"] = {"model": MODEL_NAME, "train": len(TRAIN), "dev": len(DEV), "test": len(TEST),
                   "batch": BATCH, "epochs": EPOCHS, "lr": LR, "all_silver": True}
json.dump(summary, open("phase5-results.json", "w"), indent=2, ensure_ascii=False)
print(json.dumps(summary["aggregate"], indent=2))
try:
    from google.colab import files
    files.download("phase5-results.json")
except Exception as e:
    print("download manually:", e)

## Reading the result

- `aggregate.<condition>.test_chrf_mean` +/- `test_chrf_std` is the headline
  per condition. Compare `morphbpe` and `penalty8` against `nllb_native`
  and against `nllb_zeroshot`.
- **Expect wide error bars.** 598 silver training pairs adapting a 600M
  model is deep low-resource; a MorphBPE-vs-native gap smaller than the
  across-seed std is not a real effect. That is itself a legitimate finding
  for a low-resource study.
- Send `phase5-results.json` back to Claude Code -> it lands in
  `experiments/nllb_finetune_v1/reports/` and Phase 6 (fuller eval,
  COMET if feasible) builds on it.